In [2]:

import numpy as np
import pandas as pd
import xarray as xr



In [3]:
# --- 1) load and parse the CSV (or adapt if you already have df) ---
df = pd.read_parquet("/cluster/work/climate/dnikolo/Cloud_analysis/np/20070115.1500_20070201.0000/Agg_03_T_06_00.parquet")
# columns storing Python-like lists
hist_cols = [
    "ice_frac_hist", "cot_hist", "cot_nan_frac_hist",
    "ctp_hist", "ctp_nan_frac_hist",
    "lat_hist", "lon_hist",
    "size_hist_km"
]
# convert string‐encoded lists to real Python lists

# --- 2) compute max history length ---
max_len = max(df[c].apply(len).max() for c in hist_cols)

# --- 3) prepare time coordinate: shape (n_tracks, max_len) ---
n = len(df)
time_vals = np.full((n, max_len), np.datetime64("NaT"), dtype="datetime64[ns]")
for i, (start, hist) in enumerate(zip(df["track_start_time"], df[hist_cols[0]])):
    # length of this track’s history
    L = len(hist)
    # generate timestamps spaced 15 min apart
    times = pd.date_range(start, periods=L, freq="15T").values
    time_vals[i, :L] = times

# --- 4) build 2D data arrays for each history variable ---
data_vars = {}
for c in hist_cols:
    arr = np.full((n, max_len), np.nan, dtype=float)
    for i, hist in enumerate(df[c]):
        arr[i, :len(hist)] = hist
    data_vars[c] = (("track", "time"), arr)

# you can also include scalar columns as 1D variables
for c in ["avg_cot", "avg_ctp", "avg_lat", "avg_lon",
          "start_ice_fraction", "end_ice_fraction",
          "track_length"]:
    data_vars[c] = ("track", df[c].values)

# --- 5) assemble into an xarray Dataset ---
ds = xr.Dataset(
    data_vars=data_vars,
    coords={
        "track": df.index.values,
        "time": (("track", "time"), time_vals)
    },
    attrs={
        "description": "Cloud track histories every 15 min"
    }
)

# --- 6) save to NetCDF ---
ds.to_netcdf("tracks_history.nc")


/cluster/work/climate/dnikolo/dump/ipykernel_1153553/2212316945.py:22: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  times = pd.date_range(start, periods=L, freq="15T").values
/cluster/work/climate/dnikolo/dump/ipykernel_1153553/2212316945.py:40: UserWarning: Converting non-nanosecond precision timedelta values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  ds = xr.Dataset(


ValueError: can only convert an array of size 1 to a Python scalar

In [4]:

df = pd.read_parquet("/cluster/work/climate/dnikolo/Cloud_analysis/np/20070115.1500_20070201.0000/Agg_03_T_06_00.parquet")
# 2. Identify all columns where every entry is a list or 1D‐ndarray
def is_sequence(x):
    return isinstance(x, (list, np.ndarray))
seq_cols = [col for col in df.columns
            if df[col].map(is_sequence).all()]

# 3. Determine the maximum length of those sequences
max_steps = max(df[col].map(len).max() for col in seq_cols)

# Build a "step" coordinate: 0, 15min, 30min, ..., (max_steps−1)*15min
step_offsets = pd.to_timedelta(np.arange(max_steps) * 15, unit='m')

# 4. Convert each sequence column into a 2D array, padding with NaN
data_vars = {}
for col in seq_cols:
    arr = np.full((len(df), max_steps), np.nan, dtype=float)
    for i, seq in enumerate(df[col]):
        arr[i, :len(seq)] = seq
    # DataArray dims: ("row", "step")
    data_vars[col] = (('row', 'step'), arr)

# 5. Build per-row coordinates
#    - original row index
#    - absolute track_start_time
#    - compute track_end_time = start + track_length
coords = {
    'row': df.index.values,
    'step': step_offsets,
    'track_start_time': ('row', pd.to_datetime(df['track_start_time'])),
    'track_end_time': (
        'row',
        pd.to_datetime(df['track_start_time']) + df['track_length']
    )
}

# 6. Create the xarray Dataset
ds = xr.Dataset(data_vars=data_vars, coords=coords)

# 7. Add any static (per-row) scalar columns as 1D variables, e.g. avg_lat, avg_lon:
for col in ['avg_lat', 'avg_lon', 'avg_cot', 'avg_ctp', 'glaciation_start_time', 'glaciation_end_time']:
    if col in df:
        # for datetimes, convert to pandas Timestamps
        vals = pd.to_datetime(df[col]) if 'time' in col else df[col].values
        ds[col] = ('row', vals)

# 8. Write to NetCDF
ds.to_netcdf('output.nc')


/cluster/work/climate/dnikolo/dump/ipykernel_1153553/3514718488.py:38: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  ds = xr.Dataset(data_vars=data_vars, coords=coords)
/cluster/work/climate/dnikolo/dump/ipykernel_1153553/3514718488.py:38: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or V